# MyTravels Infrastructure Runbook

This notebook is the end-to-end operational runbook for standing up the full MyTravels Kubernetes infrastructure on a local k3d cluster. Run cells top to bottom to bring everything up from scratch.

## Summary

- **Step 1 — Prerequisites**: Verify the required tools are installed (Rancher Desktop, k3d, kubectl, JupyterLab, OpenLens).
- **Step 2 — Create the Cluster**: Create the local k3d cluster (1 control plane, 3 workers) with port mappings for Traefik HTTP (8080) and PostgreSQL TCP (5432).
- **Step 3 — /etc/hosts**: Add the eleven `*.mytravels.local` hostnames to `/etc/hosts` for host-based ingress routing.
- **Step 4 — Namespace**: Create the `mytravels-default` namespace that holds all resources.
- **Step 5 — PostgreSQL**: Deploy PostgreSQL 17.6 with secret, PVC, deployment, and ClusterIP service.
- **Step 6 — Database Migrations**: Run the once-off `db-migrations` Job (cleanup initContainer + the `efbundle` migrations executable).
- **Step 7 — RabbitMQ**: Deploy RabbitMQ with the management and Prometheus plugins enabled via ConfigMap.
- **Step 8 — MinIO**: Deploy MinIO object storage pinned to agent-2 with hostPath-backed PVs.
- **Step 9 — SOLR**: Deploy Apache SOLR, the search index behind `/api/pointofinterest/search`.
- **Step 10 — Flagsmith**: Deploy Flagsmith, the feature-flag backend behind the three flags fronted by OpenFeature in `api`/`messaging`/`web`.
- **Step 11 — API**: Deploy the stateless ASP.NET Core REST API with its secret and service.
- **Step 12 — Messaging**: Deploy the stateless ASP.NET Core background worker that consumes RabbitMQ messages.
- **Step 13 — MCP**: Deploy the MCP server — the same service layer as the API, exposed as MCP tools over streamable HTTP for an MCP client/agent.
- **Step 14 — Web**: Deploy the React UI — a static Vite build served by nginx, with no secret and no PVC.
- **Step 15 — Observability**: Deploy the OTel Collector, Prometheus, Tempo, Grafana, and the postgres-exporter and cadvisor exporters.
- **Step 16 — Traefik Configuration**: Add the `postgres` TCP entrypoint to Traefik via `HelmChartConfig`.
- **Step 17 — Ingress**: Apply the ingress rules exposing RabbitMQ, MinIO, the API, Messaging, MCP, the Web UI, Grafana, Prometheus, the OTel Collector, and PostgreSQL through Traefik.
- **Step 18 — Full Stack Verification**: Confirm all pods, PVCs, services, ingresses, and URLs are healthy, that every Prometheus target is `up`, and that a trace reaches Tempo.
- **Step 19 — Diagnostics**: Pull logs and events per service when something misbehaves.
- **Step 20 — Seed Test Data (Optional)**: Bulk-upload a folder of geotagged photos as points of interest through the ingress, so the Web app and dashboards have real data. Skips itself if no photo folder is configured.
- **Step 21 — SOLR Search Verification**: Confirm the runtime-applied SOLR schema, run searches, and rebuild the index.
- **Step 22 — Teardown**: Delete all resources in reverse order and optionally the whole cluster.

## Services in this stage

| Tier | Services |
|---|---|
| Data | PostgreSQL, RabbitMQ, MinIO |
| Application | API, Messaging, MCP, Web |
| Feature Flagging | Flagsmith, Flagsmith task processor |
| Observability | OTel Collector, Prometheus, Tempo, Grafana, postgres-exporter, cadvisor |

## The application architecture  

![architecture](images/architecture.png)

---

## Step 1 — Prerequisites

Rancher Desktop, k3d, kubectl, JupyterLab, and Freelens/OpenLens install notes: [macOS](<../1-install tools (macos).md>) · [Ubuntu](<../1-install tools (ubuntu).md>) · [Windows](<../1-install tools (windows).md>).

Once Rancher Desktop is installed, open it and ensure the container engine is running before continuing.

In [ ]:
%%bash
echo "=== Docker ==="
docker --version
echo "=== k3d ==="
k3d --version
echo "=== kubectl ==="
kubectl version --client 2>/dev/null || kubectl version --client --short

---

## Step 2 — Create the Cluster

![cluster](images/k8s%20components.drawio.png)

Creates a local k3d cluster with 1 control plane node and 3 worker nodes. Traefik is bundled automatically by k3s and serves as the ingress controller.

| Flag | Meaning |
|---|---|
| `-p "8080:80@loadbalancer"` | Maps `localhost:8080` → cluster port 80 (Traefik web entrypoint) |
| `-p "5432:5432@loadbalancer"` | Maps `localhost:5432` → cluster port 5432 (Traefik postgres TCP entrypoint) |
| `--image ghcr.io/k3s-io/k3s:v1.35.3-k3s1` | Pins the k3s version for reproducible cluster creation |
| `--servers 1` | 1 control plane node (HA is meaningless on a learning cluster) |
| `--agents 3` | 3 worker nodes (the minimum this stack needs — see below) |

**Why 1 server + 3 agents?** The full stack in this lesson runs ~14 steady-state application pods across `mytravels-default`:

| Group | Pods | Notes |
|---|---|---|
| App | 6 | `api` ×2, `messaging` ×2, `mcp` ×1, `web` ×1 |
| Data | 3 | `postgres` ×1, `rabbitmq` ×1, `minio` ×1 (pinned to `agent-2`) |
| Observability | 5 | `otel-collector`, `prometheus`, `tempo`, `grafana`, `postgres-exporter` |
| DaemonSet | 1/node | `cadvisor` — one pod per node, plus k3s's `svclb-traefik` |
| Job | 1 (transient) | `migrations` — runs once and completes |

Total requested capacity is roughly **1.2 CPU / 2.4 GiB** — trivial for any node, so the constraint is not resource pressure but the **`k3d-mytravels-agent-2` nodeSelector on MinIO (Step 8)**. k3d numbers agents from `0`, so `agent-2` requires at least three agents (`agent-0`, `agent-1`, `agent-2`) to exist. Going higher just adds idle nodes: every extra agent pays ~250 MiB of k3s/containerd overhead in the Rancher Desktop VM and adds one more cadvisor + svclb-traefik DaemonSet pod for no scheduling gain. `cadvisor` still demonstrates the DaemonSet pattern across all 4 nodes (1 server + 3 agents), so nothing pedagogical is lost.

> Every port the browser uses goes through `8080`, because that is the only HTTP port mapped into the cluster. There is no second mapping per service — Traefik routes by hostname on that one port.

> Skip this cell if the cluster already exists (`k3d cluster list`).

> **Ensure Rancher Desktop is running before this cell.** k3d creates the cluster's nodes as Docker containers, so it needs a live Docker daemon — on Linux there's no system Docker install, Rancher Desktop *is* the daemon. If it isn't running (or hasn't finished starting its VM yet), `k3d cluster create` fails immediately with `Cannot connect to the Docker daemon at unix:///home/<user>/.rd/docker.sock`, because that socket file doesn't exist until Rancher Desktop creates it. Open Rancher Desktop and wait for it to fully start, then confirm with `docker info` before retrying.

In [ ]:
%%bash
export PATH="/opt/homebrew/bin:/usr/local/bin:$PATH"
k3d cluster list
echo ""
if k3d cluster list -o json | grep -q '"name":"mytravels"'; then
  echo "A cluster named 'mytravels' already exists."
  echo "Either reuse it (skip the create cell) or delete it first:"
  echo "    k3d cluster delete mytravels"
else
  echo "No 'mytravels' cluster — safe to create."
fi

In [ ]:
%%bash
export PATH="/opt/homebrew/bin:/usr/local/bin:$PATH"
if k3d cluster list mytravels >/dev/null 2>&1; then
  echo "Cluster 'mytravels' already exists, skipping creation."
else
  k3d cluster create mytravels \
    -p "8080:80@loadbalancer" \
    -p "5432:5432@loadbalancer" \
    --image ghcr.io/k3s-io/k3s:v1.35.3-k3s1 \
    --servers 1 \
    --agents 3
fi

In [ ]:
%%bash
echo "=== Cluster Info ==="
kubectl cluster-info
echo "=== Nodes ==="
kubectl get nodes
echo ""
echo "=== Traefik ==="
kubectl get pods -n kube-system -l app.kubernetes.io/name=traefik

---

## Step 3 — /etc/hosts

The ingress rules in `9-ingress.yaml` use host-based routing. Add the entries below to your hosts file so your browser resolves the hostnames to localhost. Every host in `9-ingress.yaml` needs an entry — miss one and that service is unreachable even though the pod is healthy.

```
127.0.0.1  rabbitmq.mytravels.local
127.0.0.1  minio.mytravels.local
127.0.0.1  api.mytravels.local
127.0.0.1  messaging.mytravels.local
127.0.0.1  mcp.mytravels.local
127.0.0.1  web.mytravels.local
127.0.0.1  grafana.mytravels.local
127.0.0.1  prometheus.mytravels.local
127.0.0.1  otel.mytravels.local
```

Run **one** of the next two cells depending on your OS:

- **macOS/Linux** — appends to `/etc/hosts` via `sudo`, prompting for your password.
- **Windows** — appends to `C:\Windows\System32\drivers\etc\hosts`. There's no `sudo` on Windows, so instead the cell checks whether it has Administrator privileges and writes directly if so. If not, close Jupyter/VS Code and relaunch it "as Administrator", then re-run the cell.

### macOS/Linux

In [ ]:
import subprocess
import getpass

hosts = [
    "rabbitmq.mytravels.local",
    "minio.mytravels.local",
    "api.mytravels.local",
    "messaging.mytravels.local",
    "mcp.mytravels.local",
    "web.mytravels.local",
    "grafana.mytravels.local",
    "prometheus.mytravels.local",
    "otel.mytravels.local",
    "solr.mytravels.local",
    "flagsmith.mytravels.local",
]

with open("/etc/hosts", "r") as f:
    current = f.read()

missing = [host for host in hosts if host not in current]
for host in hosts:
    if host not in missing:
        print(f"Already present: {host}")

if missing:
    password = getpass.getpass("sudo password: ")
    for host in missing:
        entry = f"127.0.0.1  {host}\n"
        result = subprocess.run(
            ["sudo", "-S", "tee", "-a", "/etc/hosts"],
            input=f"{password}\n{entry}",
            capture_output=True,
            text=True
        )
        if result.returncode == 0:
            print(f"Added: {host}")
        else:
            print(f"Failed: {host} — {result.stderr.strip()}")

### Windows

In [ ]:
import ctypes

hosts_path = r"C:\Windows\System32\drivers\etc\hosts"
hosts = [
    "rabbitmq.mytravels.local",
    "minio.mytravels.local",
    "api.mytravels.local",
    "messaging.mytravels.local",
    "mcp.mytravels.local",
    "web.mytravels.local",
    "grafana.mytravels.local",
    "prometheus.mytravels.local",
    "otel.mytravels.local",
    "solr.mytravels.local",
    "flagsmith.mytravels.local",
]

def is_admin():
    try:
        return bool(ctypes.windll.shell32.IsUserAnAdmin())
    except Exception:
        return False

if not is_admin():
    print("Not running as Administrator — the hosts file is not writable.")
    print("Close Jupyter/VS Code and relaunch it via 'Run as Administrator', then re-run this cell.")
else:
    with open(hosts_path, "r") as f:
        current = f.read()

    with open(hosts_path, "a") as f:
        for host in hosts:
            if host in current:
                print(f"Already present: {host}")
            else:
                f.write(f"127.0.0.1  {host}\n")
                print(f"Added: {host}")

---

## Step 4 — Namespace

All MyTravels resources live in the `mytravels-default` namespace. This must exist before any service manifests are applied.

In [ ]:
%%bash
kubectl apply -f manifests/1-namespace.yaml

In [ ]:
%%bash
kubectl get namespace mytravels-default

---

## Step 5 — PostgreSQL

Deploys PostgreSQL 17.6. The secret keys (`POSTGRES_USER`, `POSTGRES_PASSWORD`, `POSTGRES_DB`) match the docker-compose environment variable names exactly.

| File | Creates |
|---|---|
| `1-secret.yaml` | `postgres-secret` — DB credentials |
| `2-pvc.yaml` | `postgres-data-pvc` — 200Mi data volume |
| `3-deployment.yaml` | `postgres` deployment with liveness probe |
| `4-service.yaml` | `postgres` ClusterIP service on 5432 |


In [ ]:
%%bash
kubectl apply -f manifests/postgres

In [ ]:
%%bash
kubectl rollout status deployment/postgres -n mytravels-default
echo ""
kubectl get pods,pvc,svc -n mytravels-default -l app=postgres

---

## Step 6 — Database Migrations

Runs two once-off tasks in sequence. The `cleanup-migrations` initContainer deletes specific rows from `EFMigrationsHistory`, then the `migrate-core-db` main container runs the `efbundle` migrations executable against `CoreDbContext`. The Job completes once — it is never restarted (`restartPolicy: Never`).

| docker-compose service | Kubernetes equivalent |
|---|---|
| `cleanup-migrations` | `initContainer: cleanup-migrations` in the `db-migrations` Job |
| `migrate-core-db` | main container in the `db-migrations` Job |
| `depends_on: postgres: service_healthy` | `pg_isready` loop inside `cleanup-migrations` |
| `restart: "no"` | `restartPolicy: Never` on the Job pod |

| File | Creates |
|---|---|
| `1-secret.yaml` | `migrations-secret` — `ConnectionStrings__CoreDbContext` |
| `2-job.yaml` | `db-migrations` Job |

Postgres credentials (`POSTGRES_USER`, `POSTGRES_PASSWORD`, `POSTGRES_DB`) are pulled from the existing `postgres-secret`.

In [ ]:
%%bash
# kubectl delete job db-migrations -n mytravels-default --wait=true
kubectl apply -f manifests/migrations/

In [ ]:
%%bash
# Poll until the pod is scheduled (handles the case where the cell runs before the pod exists)
until kubectl get pod -l job-name=db-migrations -n mytravels-default 2>/dev/null | grep -q db-migrations; do
  echo "Waiting for pod to be scheduled..."; sleep 2
done

# Wait until the init container finishes (pod moves past PodInitializing)
kubectl wait pod -l job-name=db-migrations -n mytravels-default \
  --for=condition=Initialized --timeout=120s

# Wait for the job to complete — guarantees migrate-core-db has started and exited
kubectl wait job/db-migrations -n mytravels-default \
  --for=condition=Complete --timeout=300s

echo "--LIST JOBS--"
kubectl get job db-migrations -n mytravels-default
echo "--CLEANUP MIGRATION LOGS--"
kubectl logs -n mytravels-default -l job-name=db-migrations -c cleanup-migrations
echo "--MIGRATION LOGS--"
kubectl logs -n mytravels-default -l job-name=db-migrations -c migrate-core-db

---

## Step 7 — RabbitMQ

Deploys RabbitMQ 3 with the management and Prometheus plugins. `2-configmap.yaml` supplies the `enabled_plugins` file (`[rabbitmq_management,rabbitmq_prometheus].`) that activates them — without it RabbitMQ starts with 0 plugins, the management UI never comes up, and Prometheus has nothing to scrape on 15692.

`kubectl apply -f <dir>` walks a directory in filename order, which is what the numeric prefixes are for: the ConfigMap has to exist before the Deployment that mounts it, and the Secret and PVC before the pod that consumes them.

| File | Creates |
|---|---|
| `1-secret.yaml` | `rabbitmq-secret` — broker credentials |
| `2-configmap.yaml` | `rabbitmq-config` — `enabled_plugins` file |
| `3-pvc.yaml` | `rabbitmq-data-pvc` (200Mi) |
| `4-deployment.yaml` | `rabbitmq` deployment |
| `5-service.yaml` | `rabbitmq` (ClusterIP — AMQP 5672 + metrics 15692) + `rabbitmq-management` (ClusterIP 15672) |

| Port | Purpose |
|---|---|
| 5672 | AMQP — api and messaging connect here |
| 15672 | Management UI — exposed via Ingress (Step 17) |
| 15692 | Prometheus metrics — scraped in-cluster by the `rabbitmq` job (Step 15) |

The management UI is accessible at [http://rabbitmq.mytravels.local:8080](http://rabbitmq.mytravels.local:8080) via the Traefik Ingress (Step 17).

In [ ]:
%%bash
kubectl apply -f manifests/rabbitmq/

In [ ]:
%%bash
kubectl rollout status deployment/rabbitmq -n mytravels-default
echo ""
kubectl get pods,pvc,svc -n mytravels-default -l app=rabbitmq

In [ ]:
%%bash
# Confirm both plugins loaded. 'completed with 0 plugins' means the ConfigMap did not mount.
POD=$(kubectl get pod -n mytravels-default -l app=rabbitmq -o jsonpath='{.items[0].metadata.name}')
kubectl logs -n mytravels-default "$POD" --tail=40 | grep -E 'plugin|completed'
echo ""
echo "=== Enabled plugins ==="
kubectl exec -n mytravels-default "$POD" -- rabbitmq-plugins list -e

---

## Step 8 — MinIO

Deploys MinIO (S3-compatible object storage). The deployment uses a `nodeSelector` pinned to `k3d-mytravels-agent-2` and manual PVs with `hostPath` mounts on that node.

| File | Creates |
|---|---|
| `1-secret.yaml` | `minio-secret` — root credentials |
| `2-pv-pvc.yaml` | PVs + PVCs for data (200Mi) and config (50Mi) |
| `3-deployment.yaml` | `minio` deployment pinned to agent-2 |
| `4-service.yaml` | `minio` (ClusterIP 9000) + `minio-console` (ClusterIP 9090) |

The deployment also sets `MINIO_PROMETHEUS_AUTH_TYPE: public`, which lets Prometheus scrape `minio:9000/minio/v2/metrics/cluster` without a bearer token (Step 15). Fine for a local teaching cluster; in production you would issue MinIO a scrape token instead.

In [ ]:
%%bash
kubectl apply -f manifests/minio/

In [ ]:
%%bash
kubectl rollout status deployment/minio -n mytravels-default
echo ""
kubectl get pods,pvc,svc -n mytravels-default -l app=minio

---

## Step 9 — SOLR

Deploys Apache SOLR, the sole backend for POI search. Like Postgres and MinIO it is a backing
service rather than something this repo builds, so it comes up before `api`, `messaging` and
`mcp`, all three of which are configured to reach it at `http://solr:8983`.

| File | Creates |
|---|---|
| `1-pvc.yaml` | `solr-data-pvc` (1Gi) |
| `2-deployment.yaml` | `solr` deployment |
| `3-service.yaml` | `solr` (ClusterIP 8983) |

Two things differ from the MinIO manifests next door, and both are deliberate:

- **No manual PV and no `nodeSelector`.** MinIO pins itself to `k3d-mytravels-agent-2` with
  `hostPath` PVs. SOLR runs as uid 8983 rather than root, and a `hostPath` volume comes up
  root-owned, so it would fail to create its core. This uses the dynamically provisioned
  `local-path` class instead — the same one Postgres, Prometheus, Tempo and Grafana use, which
  creates the directory world-writable.
- **Liveness and readiness ask different questions.** `livenessProbe` hits
  `/solr/admin/info/system` (is SOLR up at all?) while `readinessProbe` hits
  `/solr/mytravels-pois/admin/ping` (does the collection exist and answer?). `solr-precreate`
  creates that collection on first boot, so the pod is live for a few seconds before it is ready.

The collection's *fields* are not in any manifest — `SolrSchemaInitializer` inside `messaging`
adds them through SOLR's Schema API at startup, so there is one schema definition in C# rather
than a configset duplicated into Compose, stage 3 and stage 4. Step 21 checks they landed.

The Admin UI is reachable at [http://solr.mytravels.local:8080](http://solr.mytravels.local:8080)
via the Traefik Ingress (Step 17).


In [ ]:
%%bash
kubectl apply -f manifests/solr/


In [ ]:
%%bash
kubectl rollout status deployment/solr -n mytravels-default
echo ""
kubectl get pods,pvc,svc -n mytravels-default -l app=solr


---

## Step 10 — Flagsmith

Deploys [Flagsmith](https://flagsmith.com), the feature-flag backend behind the three flags from `prompts/add feature flagging.md` (`enable-image-description`, `enable-poi-search`, `enable-message-tracing`), fronted by the OpenFeature SDKs in `api`, `messaging`, and `web`. Like the app services, it is stateless — it shares the existing `mytravels-postgres` instance in a second database (`FeatureDb`), not a second container, so there is no PVC here either.

| File | Creates |
|---|---|
| `1-secret.yaml` | `flagsmith-secret` — the Django secret key and seed admin password |
| `1b-configmap.yaml` | `flagsmith-config` — non-sensitive Django settings shared by every Flagsmith container |
| `2-create-db-job.yaml` | `flagsmith-create-db` Job — creates the `FeatureDb` database (idempotent) |
| `3-migrate-job.yaml` | `flagsmith-migrate` Job — Django migrations against `FeatureDb` |
| `3b-bootstrap-job.yaml` | `flagsmith-bootstrap` Job — creates the superuser + organisation + project (idempotent) |
| `4-deployment.yaml` | `flagsmith` deployment — the `serve` command (REST API + admin console) |
| `4b-task-processor-deployment.yaml` | `flagsmith-task-processor` deployment — background task queue |
| `4c-service.yaml` | `flagsmith` ClusterIP service on 8000 |
| `5-seed-rbac.yaml` | `flagsmith-seed` ServiceAccount/Role/RoleBinding — create/update Secrets, this namespace only |
| `6-seed-job.yaml` | `flagsmith-seed` Job — creates the "Production" environment and the three flags, then writes `flagsmith-keys` |

Postgres credentials come from the existing `postgres-secret` (Step 5) — `DATABASE_URL` is composed from it via `$(POSTGRES_USER)`/`$(POSTGRES_PASSWORD)` substitution in each Job/Deployment's env, the same technique `db-migrations` already uses for its connection string.

**Why this needs more than one `kubectl apply`.** A Job starts running the moment it is created — Kubernetes has no equivalent of Compose's `depends_on: condition: service_completed_successfully`. `flagsmith-create-db`, `flagsmith-migrate`, and `flagsmith-bootstrap` are applied together in the next cell and lean on their own retry/backoff (`backoffLimit: 5`, exponential) to absorb the resulting race: `flagsmith-migrate`'s first attempt fails fast if `FeatureDb` doesn't exist yet, then succeeds a retry or two later once `flagsmith-create-db` has finished. The three sequential `kubectl wait` calls below don't create that ordering — the Jobs' own retries do — they just block each cell until that stage has actually finished before moving on.

`flagsmith-seed` is applied as a **separate, later step**, only after waiting for `flagsmith-bootstrap` to complete. Its script *reads* the org/project bootstrap created (`Organisation.objects.get(...)`) rather than `get_or_create`-ing them, so on a `DoesNotExist` there is nothing to retry into — its much smaller `backoffLimit: 3` is not the same safety margin as the three Jobs above, so it isn't applied until bootstrap is confirmed done.

The seed Job runs the same Django-ORM script this repo's Compose stage verified by hand, but writes its two captured keys to a Kubernetes Secret (`flagsmith-keys`) instead of rewriting a bind-mounted `.env` file. Its pod has two containers: an `initContainer` running the ORM script (the Flagsmith image ships no `kubectl`), and a main container using `alpine/kubectl:1.35.0` to upsert `flagsmith-keys` from the `ServiceAccount` in `5-seed-rbac.yaml`. `api`, `messaging`, and `web` (the next three steps) all read `Flagsmith__ServerSideEnvironmentKey` / `VITE_FLAGSMITH_ENVIRONMENT_ID` from `flagsmith-keys`, so this step must finish before theirs.


In [ ]:
%%bash
kubectl apply \
  -f manifests/flagsmith/1-secret.yaml \
  -f manifests/flagsmith/1b-configmap.yaml \
  -f manifests/flagsmith/2-create-db-job.yaml \
  -f manifests/flagsmith/3-migrate-job.yaml \
  -f manifests/flagsmith/3b-bootstrap-job.yaml \
  -f manifests/flagsmith/4-deployment.yaml \
  -f manifests/flagsmith/4b-task-processor-deployment.yaml \
  -f manifests/flagsmith/4c-service.yaml


In [ ]:
%%bash
# Sequential on purpose — see the note above on why this, not the Jobs'
# creation order, is what actually enforces create-db -> migrate -> bootstrap.
for JOB in flagsmith-create-db flagsmith-migrate flagsmith-bootstrap; do
  echo "=== $JOB ==="
  until kubectl get pod -l job-name="$JOB" -n mytravels-default 2>/dev/null | grep -q "$JOB"; do
    echo "Waiting for pod to be scheduled..."; sleep 2
  done
  kubectl wait pod -l job-name="$JOB" -n mytravels-default \
    --for=condition=Initialized --timeout=120s
  kubectl wait job/"$JOB" -n mytravels-default \
    --for=condition=Complete --timeout=300s
  echo "--LOGS ($JOB)--"
  kubectl logs -n mytravels-default -l job-name="$JOB" --tail=50
  echo ""
done


In [ ]:
%%bash
kubectl rollout status deployment/flagsmith -n mytravels-default
kubectl rollout status deployment/flagsmith-task-processor -n mytravels-default
echo ""
kubectl get pods,svc -n mytravels-default -l app=flagsmith
kubectl get pods -n mytravels-default -l app=flagsmith-task-processor


In [ ]:
%%bash
# Applied only now, after flagsmith-bootstrap (previous cell) has completed —
# see the note above on why this Job doesn't get the same "apply everything up
# front" treatment as create-db/migrate/bootstrap.
kubectl apply -f manifests/flagsmith/5-seed-rbac.yaml -f manifests/flagsmith/6-seed-job.yaml


In [ ]:
%%bash
until kubectl get pod -l job-name=flagsmith-seed -n mytravels-default 2>/dev/null | grep -q flagsmith-seed; do
  echo "Waiting for pod to be scheduled..."; sleep 2
done
kubectl wait pod -l job-name=flagsmith-seed -n mytravels-default \
  --for=condition=Initialized --timeout=120s
kubectl wait job/flagsmith-seed -n mytravels-default \
  --for=condition=Complete --timeout=180s
echo "--ORM SCRIPT LOGS (initContainer flagsmith-shell)--"
kubectl logs -n mytravels-default -l job-name=flagsmith-seed -c flagsmith-shell --tail=50
echo "--SECRET UPSERT LOGS (kubectl-apply)--"
kubectl logs -n mytravels-default -l job-name=flagsmith-seed -c kubectl-apply --tail=50

echo ""
echo "=== flagsmith-keys secret ==="
if kubectl get secret flagsmith-keys -n mytravels-default >/dev/null 2>&1; then
  kubectl get secret flagsmith-keys -n mytravels-default \
    -o go-template='{{range $k, $v := .data}}{{$k}} {{end}}'
  echo ""
else
  echo "FAILED — flagsmith-keys secret was not created. Diagnosing inline:"
  kubectl describe job/flagsmith-seed -n mytravels-default | sed -n '/Events:/,$p'
  kubectl logs -n mytravels-default -l job-name=flagsmith-seed -c kubectl-apply --tail=100
fi


---

## Step 11 — API

Deploys the MyTravels ASP.NET Core REST API. The service is stateless — no PVC is required.

Secret keys match the docker-compose `environment` keys exactly. Non-sensitive config (`ASPNETCORE_ENVIRONMENT`, `ASPNETCORE_URLS`, public URLs, `MinIO__Endpoint`, the OTel settings) is set directly in the Deployment rather than in the Secret.

| docker-compose env var | Kubernetes |
|---|---|
| `ConnectionStrings__CoreDbContext` | Secret `api-secret` → `secretKeyRef` |
| `GoogleApiKey` | Secret `api-secret` → `secretKeyRef` |
| `RabbitMQ__Uri` | Secret `api-secret` → `secretKeyRef` |
| `MinIO__AccessKey` | Secret `api-secret` → `secretKeyRef` |
| `MinIO__SecretKey` | Secret `api-secret` → `secretKeyRef` |
| `ASPNETCORE_ENVIRONMENT` | Direct env var `Production` |
| `ASPNETCORE_URLS` | Direct env var `http://+:5101` |
| `GoogleMapsUrl` | Direct env var `https://maps.googleapis.com` |
| `GooglePlacesUrl` | Direct env var `https://places.googleapis.com` |
| `MinIO__Endpoint` | Direct env var `minio:9000` (in-cluster DNS) |
| `OTEL_EXPORTER_OTLP_ENDPOINT` | Direct env var `http://otel-collector:4317` (Step 15) |
| `OTEL_SERVICE_NAME` | Direct env var `mytravels-api` (Step 15) |

| File | Creates |
|---|---|
| `1-secret.yaml` | `api-secret` — credentials and tokens |
| `2-deployment.yaml` | `api` deployment with liveness probe on `/health` |
| `3-service.yaml` | `api` ClusterIP service on 5101 |

In [ ]:
%%bash
kubectl apply -f manifests/api/

In [ ]:
%%bash
kubectl rollout status deployment/api -n mytravels-default
echo ""
kubectl get pods,svc -n mytravels-default -l app=api

---

## Step 12 — Messaging

Deploys the MyTravels ASP.NET Core background worker that consumes RabbitMQ messages. The service is stateless — no PVC is required.

Secret keys match the docker-compose `environment` keys exactly. Non-sensitive config (`ASPNETCORE_ENVIRONMENT`, `ASPNETCORE_URLS`, public URLs, `MinIO__Endpoint`, the OTel settings) is set directly in the Deployment. Shared secrets (`ConnectionStrings__CoreDbContext`, `RabbitMQ__Uri`, `MinIO__AccessKey`, `MinIO__SecretKey`, `GoogleApiKey`) use the same base64 values as `api-secret` but are stored in the dedicated `messaging-secret`.

| docker-compose env var | Kubernetes |
|---|---|
| `ConnectionStrings__CoreDbContext` | Secret `messaging-secret` → `secretKeyRef` |
| `RabbitMQ__Uri` | Secret `messaging-secret` → `secretKeyRef` |
| `MinIO__AccessKey` | Secret `messaging-secret` → `secretKeyRef` |
| `MinIO__SecretKey` | Secret `messaging-secret` → `secretKeyRef` |
| `GoogleApiKey` | Secret `messaging-secret` → `secretKeyRef` |
| `AnthropicModel` | ConfigMap `messaging-config` → `configMapKeyRef` |
| `ASPNETCORE_ENVIRONMENT` | Direct env var `Production` |
| `ASPNETCORE_URLS` | Direct env var `http://+:5102` |
| `GoogleMapsUrl` | Direct env var `https://maps.googleapis.com` |
| `GooglePlacesUrl` | Direct env var `https://places.googleapis.com` |
| `MinIO__Endpoint` | Direct env var `minio:9000` (in-cluster DNS) |
| `OTEL_EXPORTER_OTLP_ENDPOINT` | Direct env var `http://otel-collector:4317` (Step 15) |
| `OTEL_SERVICE_NAME` | Direct env var `mytravels-messaging` (Step 15) |

| File | Creates |
|---|---|
| `1-secret.yaml` | `messaging-secret` — credentials and tokens |
| `2-configmap.yaml` | `messaging-config` — non-secret config (Anthropic model name) |
| `3-deployment.yaml` | `messaging` deployment with liveness probe on `/health` |
| `4-service.yaml` | `messaging` ClusterIP service on 5102 |

The messaging worker has no user-facing UI, but it is exposed via Ingress at [http://messaging.mytravels.local:8080/health](http://messaging.mytravels.local:8080/health) for health checks (Step 17). Its work is observed through Grafana instead — traces under `service.name=mytravels-messaging` in Tempo, and queue depth via the RabbitMQ metrics Prometheus scrapes on 15692.

> **Dependency:** RabbitMQ (Step 7) must be healthy and MinIO (Step 8) must be running before the messaging worker can process messages.

In [ ]:
%%bash
kubectl apply -f manifests/messaging/

In [ ]:
%%bash
kubectl rollout status deployment/messaging -n mytravels-default
echo ""
kubectl get pods,svc -n mytravels-default -l app=messaging

---

## Step 13 — MCP

Deploys the MyTravels MCP server — exposes `upload_photo`, `upload_photo_with_coordinates`, and `search_place` as MCP tools over streamable HTTP, reusing the same service layer as the API rather than proxying it. The service is stateless — no PVC is required.

Secret keys match the docker-compose `environment` keys exactly, and reuse the same base64 values as `api-secret`. Non-sensitive config is set directly in the Deployment, same pattern as API/Messaging. Unlike API and Messaging, there is no `GoogleApiKey` entry at all — MCP's `search_place` tool still calls `IMapsService`, but geocoding falls back to OpenStreetMap without it, matching the API's existing behavior in this stage (SPEC F-14).

| docker-compose env var | Kubernetes |
|---|---|
| `ConnectionStrings__CoreDbContext` | Secret `mcp-secret` → `secretKeyRef` |
| `RabbitMQ__Uri` | Secret `mcp-secret` → `secretKeyRef` |
| `MinIO__AccessKey` | Secret `mcp-secret` → `secretKeyRef` |
| `MinIO__SecretKey` | Secret `mcp-secret` → `secretKeyRef` |
| `ASPNETCORE_ENVIRONMENT` | Direct env var `Production` |
| `ASPNETCORE_URLS` | Direct env var `http://+:5103` |
| `GoogleMapsUrl` | Direct env var `https://maps.googleapis.com` |
| `GooglePlacesUrl` | Direct env var `https://places.googleapis.com` |
| `MinIO__Endpoint` | Direct env var `minio:9000` (in-cluster DNS) |
| `OTEL_EXPORTER_OTLP_ENDPOINT` | Direct env var `http://otel-collector:4317` (Step 15) |
| `OTEL_SERVICE_NAME` | Direct env var `mytravels-mcp` (Step 15) |

| File | Creates |
|---|---|
| `1-secret.yaml` | `mcp-secret` — credentials and tokens |
| `2-deployment.yaml` | `mcp` deployment with liveness probe on `/health` (no Swagger here, unlike API) |
| `3-service.yaml` | `mcp` ClusterIP service on 5103 |

**Endpoints.** `Program.cs` maps only two things: `GET /health` (used by the probes and by the verification cell in Step 18) and `MapMcp()` at the root path, which is the streamable-HTTP MCP transport — it speaks JSON-RPC over `POST /`, so opening it in a browser is not a useful check.

**Reaching it.** `9-ingress.yaml` (Step 17) routes `mcp.mytravels.local` → `mcp:5103`, so MCP is reachable both ways:

- in-cluster: `http://mcp:5103` — how another pod would call it
- from your machine: `http://mcp.mytravels.local:8080` — through Traefik, the same as every other host-routed service

> **`.mcp.json` in this directory points at `http://mcp.mytravels.local/` (port 80).** The k3d cluster maps host port **8080** → cluster port 80 (Step 2), so a client on your machine needs `http://mcp.mytravels.local:8080/`. Either fix the URL in `.mcp.json`, or recreate the cluster with `-p "80:80@loadbalancer"` if you want the bare hostname to work.


In [ ]:
%%bash
kubectl apply -f manifests/mcp/

In [ ]:
%%bash
kubectl rollout status deployment/mcp -n mytravels-default
echo ""
kubectl get pods,svc -n mytravels-default -l app=mcp

---

## Step 14 — Web

Deploys the MyTravels React UI. The image is a static Vite build served by nginx (see [src/web/Dockerfile](<../src/web/Dockerfile>)) — no PVC, but it does carry a ConfigMap and a Secret, because the `VITE_*` values baked into the bundle are substituted at pod start.

| docker-compose | Kubernetes |
|---|---|
| `image: tshepontlhokoa/mytravels-web:v1.0.9` | `tshepontlhokoa/mytravels-web:v1.0.9` |
| `ports: 5100:80` | `web` ClusterIP service on 80, reached through the Ingress (Step 17) |
| `build.args.VITE_API_BASE_URL` | `web-config` ConfigMap, substituted into the bundle by an init container — see the note below |
| `build.args.VITE_OTEL_EXPORTER_OTLP_TRACES_ENDPOINT` | same — `web-config` ConfigMap |
| `build.args.VITE_FLAGSMITH_API_URL` | same — `web-config` ConfigMap |
| `build.args.VITE_FLAGSMITH_ENVIRONMENT_ID` | `flagsmith-keys` Secret instead — it's Flagsmith-generated, not hand-configured (see Step 10) |
| `depends_on: api` | no equivalent; the pod starts regardless and the browser's API calls fail until the API is up |

| File | Creates |
|---|---|
| `1-configmap.yaml` | `web-config` ConfigMap holding the three browser-facing URLs |
| `2-deployment.yaml` | `web` deployment with a `render-config` init container, plus readiness and liveness probes on `/` |
| `3-service.yaml` | `web` ClusterIP service on 80 |

> **`VITE_API_BASE_URL` is a build-time argument, not a runtime one.** Vite inlines it into the JavaScript bundle when the image is built, so unlike every other service in this runbook there is no env var nginx could read to change it. The same goes for `VITE_OTEL_EXPORTER_OTLP_TRACES_ENDPOINT` — and that one is stricter still: `telemetry.ts` wraps its OpenTelemetry setup in `if (otlpEndpoint)`, so leaving the build arg unset makes the branch statically false and Vite drops the entire browser-RUM block from the bundle. You cannot patch in code that was never emitted. `VITE_FLAGSMITH_API_URL` has a subtler failure mode: leaving it unset doesn't break the build, but the Flagsmith JS SDK then falls back to its public SaaS default (`edge.api.flagsmith.com`), which rejects this self-hosted environment's key with `403`.
>
> **What this step does about it.** The `v1.0.9` image is built with sentinel placeholders instead of real values — `VITE_API_BASE_URL=__API_BASE_URL__`, `VITE_OTEL_EXPORTER_OTLP_TRACES_ENDPOINT=__OTLP_TRACES_ENDPOINT__`, `VITE_FLAGSMITH_API_URL=__FLAGSMITH_API_URL__`, and `VITE_FLAGSMITH_ENVIRONMENT_ID=__FLAGSMITH_ENVIRONMENT_ID__` (see [2-dockerhub/docker-compose.build.yml](<../2-dockerhub/docker-compose.build.yml>)). All four are non-empty, so all the code survives the build, and all land in the bundle as ordinary strings. The Deployment then adds a `render-config` init container that copies the site out of the read-only image layer into an `emptyDir`, `sed`s the sentinels to the values held in the `web-config` ConfigMap and the `flagsmith-keys` Secret, and mounts that `emptyDir` over nginx's docroot. If a sentinel somehow survives, the init container exits non-zero rather than serve a page that would call `__API_BASE_URL__` from the browser.
>
> The payoff: changing a URL is now an edit to `1-configmap.yaml` and a `kubectl rollout restart deployment/web -n mytravels-default`. No rebuild, no push, no tag bump. All three ConfigMap-sourced values are resolved by the **browser**, not by a pod, so they must be ingress hostnames the host can reach — `http://api.mytravels.local:8080`, not `http://api:5101` — including `VITE_FLAGSMITH_API_URL`, which points at the Flagsmith ingress (`http://flagsmith.mytravels.local:8080/api/v1/`), not the in-cluster `flagsmith:8000` service `api`/`messaging` use.
>
> This is one of the two standard answers to build-time config in a container image; the other is to have the entrypoint write a `config.js` that the app reads at load time, which needs a change to the application source. Substituting into the already-built bundle needs nothing but the image and a ConfigMap.

In [ ]:
%%bash
kubectl apply -f manifests/web/

In [ ]:
%%bash
kubectl rollout status deployment/web -n mytravels-default
echo ""
kubectl get pods,svc -n mytravels-default -l app=web

---

## Step 15 — Observability

Deploys metrics and distributed tracing for the whole stack. The API, messaging worker, and MCP server push OTLP metrics/traces to the OTel Collector, which exposes a Prometheus scrape endpoint and forwards traces to Tempo. Prometheus also scrapes RabbitMQ, MinIO, and postgres-exporter directly (they already speak the Prometheus exposition format), plus cadvisor for per-container/node metrics. Grafana ships with both datasources and one starter dashboard pre-provisioned.

```
api / messaging / mcp ──OTLP:4317──► otel-collector ──:8889 (scraped)──► prometheus ──► grafana
                                          └──────traces──────► tempo ──────────────────► grafana
rabbitmq :15692 ─┐
minio    :9000  ─┼──scraped──────────────────────────────────► prometheus
postgres-exporter :9187 ─┤
cadvisor :8080 (DaemonSet, one per node) ─┘
```

### Components

| Component | Image | Port(s) | Role |
|---|---|---|---|
| OTel Collector | `otel/opentelemetry-collector-contrib:0.116.1` | 4317 gRPC, 4318 HTTP, 8889 metrics | Receives OTLP from the .NET services; exposes `:8889` for Prometheus, forwards traces to Tempo |
| Prometheus | `prom/prometheus:v3.1.0` | 9090 | Scrapes every target below; 1Gi PVC for the TSDB |
| Tempo | `grafana/tempo:2.6.1` | 4317/4318 OTLP in, 3200 query API | Trace storage and query; 1Gi PVC |
| Grafana | `grafana/grafana:11.4.0` | 3000 | Dashboards over Prometheus + Tempo; 200Mi PVC; sign-up disabled |
| postgres-exporter | `prometheuscommunity/postgres-exporter:v0.15.0` | 9187 | PostgreSQL metrics — reuses `postgres-secret`, no new credential |
| cadvisor | `gcr.io/cadvisor/cadvisor:v0.49.1` | 8080 | Container/node resource metrics — **DaemonSet**, one pod per node |

### Prometheus scrape targets (`4-prometheus-configmap.yaml`)

| Job | Target | Source |
|---|---|---|
| `prometheus` | `localhost:9090` | itself |
| `otel-collector` | `otel-collector:8889` | api, messaging, mcp (via OTLP) |
| `postgres` | `postgres-exporter:9187` | PostgreSQL |
| `rabbitmq` | `rabbitmq:15692` | `rabbitmq_prometheus` plugin |
| `minio` | `minio:9000` | MinIO built-in `/minio/v2/metrics/cluster` |
| `cadvisor` | DNS `SRV` on the headless `cadvisor` Service | one entry per node, discovered dynamically |

cadvisor is the one job that isn't a fixed target: because it's a DaemonSet, the pod count changes with the node count, so Prometheus uses DNS `SRV` discovery against the headless (`clusterIP: None`) `cadvisor` Service instead of a hardcoded list. Add a node, and the new cadvisor pod is scraped without editing any config.

### Files

| File | Creates |
|---|---|
| `1-otel-collector-configmap.yaml` … `3-otel-collector-service.yaml` | ConfigMap, Deployment, Service |
| `4-prometheus-configmap.yaml` … `7-prometheus-service.yaml` | ConfigMap, PVC, Deployment, Service |
| `8-tempo-configmap.yaml` … `11-tempo-service.yaml` | ConfigMap, PVC, Deployment, Service |
| `12-postgres-exporter-deployment.yaml`, `13-postgres-exporter-service.yaml` | Deployment, Service |
| `14-cadvisor-daemonset.yaml`, `15-cadvisor-service.yaml` | DaemonSet, headless Service |
| `16-grafana-secret.yaml` … `21-grafana-service.yaml` | Secret, datasources ConfigMap, dashboards ConfigMap, PVC, Deployment, Service |

### Changes to earlier manifests

This step also changes manifests already applied in earlier steps — the cell below re-applies them:

- `rabbitmq/2-configmap.yaml` — enables the `rabbitmq_prometheus` plugin
- `rabbitmq/4-deployment.yaml` / `5-service.yaml` — expose port `15692`
- `minio/3-deployment.yaml` — adds `MINIO_PROMETHEUS_AUTH_TYPE: public` (otherwise the metrics endpoint requires a bearer token)
- `api/2-deployment.yaml` / `messaging/2-deployment.yaml` — add `OTEL_EXPORTER_OTLP_ENDPOINT` and `OTEL_SERVICE_NAME`

MCP's `2-deployment.yaml` (Step 13) already ships with `OTEL_EXPORTER_OTLP_ENDPOINT`/`OTEL_SERVICE_NAME` baked in from creation, so it isn't part of this re-apply list — there's nothing to retrofit.

> **cadvisor needs privileged access to the node** (`hostPath` mounts of `/`, `/sys`, `/var/run`, `/var/lib/docker`) to read container metrics — this is normal for a node-level metrics agent, not a misconfiguration.

In [ ]:
%%bash
kubectl apply -f manifests/observability/
echo ""
echo "=== Re-applying manifests changed for observability ==="
kubectl apply -f manifests/rabbitmq/
kubectl apply -f manifests/minio/
kubectl apply -f manifests/api/
kubectl apply -f manifests/messaging/

In [ ]:
%%bash
kubectl rollout status deployment/otel-collector -n mytravels-default
kubectl rollout status deployment/prometheus -n mytravels-default
kubectl rollout status deployment/tempo -n mytravels-default
kubectl rollout status deployment/postgres-exporter -n mytravels-default
kubectl rollout status daemonset/cadvisor -n mytravels-default
kubectl rollout status deployment/grafana -n mytravels-default
echo ""
kubectl get pods,pvc,svc -n mytravels-default -l 'app in (otel-collector,prometheus,tempo,postgres-exporter,cadvisor,grafana)'

---

## Step 16 — Traefik Configuration

Adds the `postgres` TCP entrypoint to Traefik so that the `IngressRouteTCP` in `9-ingress.yaml` can route raw TCP connections on port 5432 to the postgres pod. Without this, connections to `127.0.0.1:5432` are refused even when postgres is healthy.

The `HelmChartConfig` patches the k3s-managed Traefik Helm release — k3s picks it up and restarts Traefik automatically within ~15 seconds.

| File | Creates |
|---|---|
| `8-traefik-config.yaml` | `HelmChartConfig/traefik` — adds `--entrypoints.postgres.address=:5432/tcp` |

> **Cluster port mapping:** the k3d cluster must have been created with `-p "5432:5432@loadbalancer"` (see Step 2) so that the Traefik entrypoint is reachable from the host.

In [ ]:
%%bash
kubectl apply -f manifests/8-traefik-config.yaml
echo ""
echo "Waiting for Traefik to restart..."
sleep 20
kubectl rollout status deploy/traefik -n kube-system --timeout=60s
echo ""
for i in $(seq 1 12); do
  ARGS=$(kubectl get deploy traefik -n kube-system -o jsonpath='{.spec.template.spec.containers[0].args}' | tr ',' '\n')
  if echo "$ARGS" | grep -q postgres; then
    echo "postgres entrypoint confirmed:"
    echo "$ARGS" | grep postgres
    break
  fi
  echo "Waiting for postgres entrypoint... ($i/12)"
  sleep 5
done

---

## Step 17 — Ingress

In [ ]:
%%bash
kubectl apply -f manifests/9-ingress.yaml

In [ ]:
%%bash
kubectl get ingress -n mytravels-default


Applies the top-level ingress rules that expose services through Traefik.

| Host | Routes to | Port | Protocol |
|---|---|---|---|
| [http://rabbitmq.mytravels.local:8080](http://rabbitmq.mytravels.local:8080) | `rabbitmq-management` service | 15672 | HTTP |
| [http://minio.mytravels.local:8080](http://minio.mytravels.local:8080) | `minio-console` service | 9090 | HTTP |
| [http://api.mytravels.local:8080](http://api.mytravels.local:8080/swagger) | `api` service | 5101 | HTTP |
| [http://messaging.mytravels.local:8080/health](http://messaging.mytravels.local:8080/health) | `messaging` service | 5102 | HTTP |
| [http://mcp.mytravels.local:8080/health](http://mcp.mytravels.local:8080/health) | `mcp` service | 5103 | HTTP (MCP streamable HTTP at `/`) |
| [http://web.mytravels.local:8080](http://web.mytravels.local:8080) | `web` service | 80 | HTTP |
| [http://grafana.mytravels.local:8080](http://grafana.mytravels.local:8080) | `grafana` service | 3000 | HTTP |
| [http://prometheus.mytravels.local:8080](http://prometheus.mytravels.local:8080) | `prometheus` service | 9090 | HTTP |
| [http://otel.mytravels.local:8080](http://otel.mytravels.local:8080) | `otel-collector` service | 4318 | HTTP (OTLP, browser RUM) |
| [http://solr.mytravels.local:8080](http://solr.mytravels.local:8080/solr/#/mytravels-pois) | `solr` service | 8983 | HTTP (Admin UI + query API) |
| [http://flagsmith.mytravels.local:8080](http://flagsmith.mytravels.local:8080) | `flagsmith` service | 8000 | HTTP (admin console + REST API) |
| `127.0.0.1:5432` (TCP) | `postgres` service | 5432 | TCP |

Three services deliberately have **no** ingress row and stay cluster-internal:

- **Tempo** (`tempo:3200`) — queried by Grafana through its datasource, not by you directly. Use Grafana's Explore view for traces.
- **postgres-exporter** (`postgres-exporter:9187`) and **cadvisor** (`cadvisor:8080`) — scrape endpoints Prometheus reads in-cluster; there's no UI to expose.

`9-ingress.yaml` defines five `Ingress` objects (`management-ingress` covering RabbitMQ/MinIO/Grafana/Prometheus/OTel/SOLR/Flagsmith, plus `api-ingress`, `messaging-ingress`, `web-ingress`, `mcp-ingress`) and one Traefik `IngressRouteTCP` for PostgreSQL.

Traffic flow (HTTP): `localhost:8080` → k3d load balancer → Traefik (port 80) → service backend.
PostgreSQL: `localhost:5432` → k3d load balancer → Traefik (postgres entrypoint) → postgres service.

---

## Step 18 — Full Stack Verification

Run these cells to confirm all resources are healthy before using the stack.

In [ ]:
%%bash
echo "=== Nodes ==="
kubectl get nodes
echo ""
echo "=== Pods ==="
kubectl get pods -n mytravels-default -o wide
echo ""
echo "=== Deployments / DaemonSets / Jobs ==="
kubectl get deployments,daemonsets,jobs -n mytravels-default
echo ""
echo "=== PVCs ==="
kubectl get pvc -n mytravels-default
echo ""
echo "=== Services ==="
kubectl get svc -n mytravels-default
echo ""
echo "=== Ingress ==="
kubectl get ingress,ingressroutetcp -n mytravels-default
echo ""

# Any pod not Running/Completed gets its events dumped here, inline.
BAD=$(kubectl get pods -n mytravels-default --no-headers \
  | awk '$3 != "Running" && $3 != "Completed" {print $1}')
if [ -n "$BAD" ]; then
  echo "=== Unhealthy pods ==="
  for POD in $BAD; do
    echo "--- $POD ---"
    kubectl describe pod "$POD" -n mytravels-default | sed -n '/Events:/,$p'
  done
else
  echo "All pods Running or Completed."
fi

In [ ]:
%%bash
# Ingress-exposed services. A check fails on any non-2xx/3xx status (or no response
# at all), and the diagnostics for that service run immediately, inline.
check() {
  NAME=$1; URL=$2; LABEL=$3
  CODE=$(curl -s -o /dev/null -w "%{http_code}" --max-time 10 "$URL" || echo "000")
  case "$CODE" in
    2*|3*) printf "%-16s %s OK\n" "$NAME" "$CODE" ;;
    *)
      printf "%-16s %s FAILED (%s)\n" "$NAME" "$CODE" "$URL"
      echo "--- pods (app=$LABEL) ---"
      kubectl get pods -n mytravels-default -l app="$LABEL" -o wide
      echo "--- logs (app=$LABEL, last 20) ---"
      kubectl logs -n mytravels-default -l app="$LABEL" --tail=20 2>&1 | sed 's/^/    /'
      echo ""
      ;;
  esac
}

echo "=== Application tier ==="
check "Web"        http://web.mytravels.local:8080                 web
check "API"        http://api.mytravels.local:8080/swagger/index.html api
check "Messaging"  http://messaging.mytravels.local:8080/health    messaging
check "MCP"        http://mcp.mytravels.local:8080/health          mcp
echo ""
echo "=== Data tier ==="
check "RabbitMQ"   http://rabbitmq.mytravels.local:8080            rabbitmq
check "MinIO"      http://minio.mytravels.local:8080               minio
check "SOLR"       http://solr.mytravels.local:8080/solr/mytravels-pois/admin/ping solr
check "Flagsmith"  http://flagsmith.mytravels.local:8080/health/liveness    flagsmith
echo ""
echo "=== Observability tier ==="
check "Grafana"    http://grafana.mytravels.local:8080/api/health   grafana
check "Prometheus" http://prometheus.mytravels.local:8080/-/healthy prometheus
echo ""

# Cluster-internal services with no ingress route — one throwaway curl pod checks them all.
echo "=== Cluster-internal endpoints ==="
kubectl run cluster-healthcheck -n mytravels-default --rm -i --restart=Never \
  --image=curlimages/curl:8.11.0 --quiet -- sh -c '
for T in "tempo|http://tempo:3200/ready" \
         "otel-collector|http://otel-collector:8889/metrics" \
         "postgres-exporter|http://postgres-exporter:9187/metrics" \
         "cadvisor|http://cadvisor:8080/healthz" \
         "mcp (in-cluster)|http://mcp:5103/health"; do
  NAME=${T%%|*}; URL=${T#*|}
  printf "%-20s %s\n" "$NAME" "$(curl -s -o /dev/null -w "%{http_code}" --max-time 10 "$URL" || echo 000)"
done'

In [ ]:
%%bash
GRAFANA_AUTH="user123:password123"   # from observability/16-grafana-secret.yaml

echo "=== Prometheus scrape targets ==="
curl -s http://prometheus.mytravels.local:8080/api/v1/targets \
  | python3 -c "
import json, sys
data = json.load(sys.stdin)
targets = data['data']['activeTargets']
down = [t for t in targets if t['health'] != 'up']
for t in targets:
    print(f\"  {t['labels'].get('job'):<18} {t['health']:<8} {t['scrapeUrl']} {t.get('lastError','')}\")
print()
print(f'{len(targets) - len(down)}/{len(targets)} targets up')
jobs = {t['labels'].get('job') for t in targets}
missing = {'prometheus','otel-collector','postgres','rabbitmq','minio','cadvisor'} - jobs
if missing:
    print('MISSING JOBS:', ', '.join(sorted(missing)))
"
echo ""

echo "=== cadvisor: one target per node (DaemonSet) ==="
NODES=$(kubectl get nodes --no-headers | wc -l | tr -d ' ')
CADVISOR_PODS=$(kubectl get pods -n mytravels-default -l app=cadvisor --no-headers | grep -c Running)
CADVISOR_TARGETS=$(curl -s http://prometheus.mytravels.local:8080/api/v1/targets \
  | python3 -c "import json,sys; print(sum(1 for t in json.load(sys.stdin)['data']['activeTargets'] if t['labels'].get('job')=='cadvisor'))")
echo "nodes=$NODES  running cadvisor pods=$CADVISOR_PODS  prometheus cadvisor targets=$CADVISOR_TARGETS"
if [ "$NODES" != "$CADVISOR_PODS" ]; then
  echo "MISMATCH — DaemonSet is not on every node:"
  kubectl get pods -n mytravels-default -l app=cadvisor -o wide
  kubectl describe daemonset cadvisor -n mytravels-default | tail -20
fi
echo ""

echo "=== Generating traces (api, then mcp) ==="
curl -s -o /dev/null -w "  api HTTP %{http_code}\n" http://api.mytravels.local:8080/api/pointofinterest
curl -s -o /dev/null -w "  mcp HTTP %{http_code}\n" http://mcp.mytravels.local:8080/health
echo ""

echo "=== Waiting for traces to reach Tempo (via the Grafana datasource proxy) ==="
for SERVICE in mytravels-api mytravels-mcp; do
  FOUND=0
  for i in $(seq 1 12); do
    COUNT=$(curl -s -u "$GRAFANA_AUTH" \
      "http://grafana.mytravels.local:8080/api/datasources/proxy/uid/tempo/api/search?tags=service.name%3D${SERVICE}&limit=1" \
      | python3 -c "import json,sys; print(len(json.load(sys.stdin).get('traces', [])))" 2>/dev/null || echo 0)
    if [ "$COUNT" -gt 0 ]; then
      echo "  $SERVICE — trace found"
      FOUND=1
      break
    fi
    sleep 5
  done
  if [ "$FOUND" -eq 0 ]; then
    echo "  $SERVICE — NO TRACE after 60s"
    echo "  --- otel-collector logs ---"
    kubectl logs -n mytravels-default -l app=otel-collector --tail=20 | sed 's/^/      /'
    echo "  --- tempo logs ---"
    kubectl logs -n mytravels-default -l app=tempo --tail=20 | sed 's/^/      /'
  fi
done

In [ ]:
%%bash
kubectl top pods -A


**Services deployed:**

| Service | Workload | Purpose |
|---|---|---|
| PostgreSQL | Deployment | Primary database |
| RabbitMQ | Deployment | Message broker (management + Prometheus plugins) |
| MinIO | Deployment | S3-compatible object storage |
| SOLR | Deployment | Search index for points of interest |
| db-migrations | Job | Once-off EF Core migrations (Step 6) |
| API | Deployment | ASP.NET Core REST API |
| Messaging | Deployment | ASP.NET Core background worker (RabbitMQ consumer) |
| MCP | Deployment | MCP server — same service layer as the API, exposed as MCP tools |
| Web | Deployment | React UI (static build served by nginx) |
| OTel Collector | Deployment | Receives OTLP metrics/traces, exports to Prometheus + Tempo |
| Prometheus | Deployment | Metrics storage and scraping |
| Tempo | Deployment | Distributed trace storage |
| Grafana | Deployment | Dashboards over Prometheus + Tempo |
| postgres-exporter | Deployment | PostgreSQL → Prometheus metrics bridge |
| cadvisor | **DaemonSet** | Per-node container resource metrics |

**Browser-reachable URLs (after full setup):**

| Service | URL | Credentials |
|---|---|---|
| Web UI | [http://web.mytravels.local:8080](http://web.mytravels.local:8080) | — |
| API (Swagger) | [http://api.mytravels.local:8080/swagger](http://api.mytravels.local:8080/swagger) | — |
| Messaging health | [http://messaging.mytravels.local:8080/health](http://messaging.mytravels.local:8080/health) | — |
| MCP health | [http://mcp.mytravels.local:8080/health](http://mcp.mytravels.local:8080/health) | — |
| RabbitMQ Management | [http://rabbitmq.mytravels.local:8080](http://rabbitmq.mytravels.local:8080) | see `rabbitmq/1-secret.yaml` |
| MinIO Console | [http://minio.mytravels.local:8080](http://minio.mytravels.local:8080) | see `minio/1-secret.yaml` |
| Grafana | [http://grafana.mytravels.local:8080](http://grafana.mytravels.local:8080) | `user123` / `password123` |
| SOLR Admin | [http://solr.mytravels.local:8080](http://solr.mytravels.local:8080) | — |
| Prometheus | [http://prometheus.mytravels.local:8080](http://prometheus.mytravels.local:8080) | — |

**Non-browser endpoints:**

| Service | Endpoint | Used by |
|---|---|---|
| MCP (streamable HTTP) | `http://mcp.mytravels.local:8080/` | MCP clients — see `.mcp.json` and the port note in Step 13 |
| OTLP ingest | `http://otel.mytravels.local:8080` (HTTP) / `otel-collector:4317` (gRPC) | api, messaging, mcp, browser RUM |
| Tempo query API | `tempo:3200` (in-cluster only) | Grafana's Tempo datasource |
| postgres-exporter | `postgres-exporter:9187` (in-cluster only) | Prometheus |
| cadvisor | `cadvisor:8080` (in-cluster, headless) | Prometheus (DNS `SRV` discovery) |
| PostgreSQL | `127.0.0.1:5432` (TCP via Traefik) | psql, DBeaver, any SQL client |


---

## Step 19 — Diagnostics

Run these cells when a service is not behaving as expected.

In [ ]:
%%bash
echo "=== PostgreSQL logs ==="
kubectl logs -n mytravels-default -l app=postgres --tail=50

In [ ]:
%%bash
echo "=== RabbitMQ logs ==="
kubectl logs -n mytravels-default -l app=rabbitmq --tail=50

In [ ]:
%%bash
echo "=== MinIO logs ==="
kubectl logs -n mytravels-default -l app=minio --tail=50

In [ ]:
%%bash
echo "=== SOLR logs ==="
kubectl logs -n mytravels-default -l app=solr --tail=50


In [ ]:
%%bash
echo "=== Flagsmith (serve) logs ==="
kubectl logs -n mytravels-default -l app=flagsmith --tail=50
echo ""
echo "=== Flagsmith task-processor logs ==="
kubectl logs -n mytravels-default -l app=flagsmith-task-processor --tail=50
echo ""
echo "=== Flagsmith Jobs (create-db / migrate / bootstrap / seed) ==="
kubectl get jobs -n mytravels-default -l 'job-name in (flagsmith-create-db,flagsmith-migrate,flagsmith-bootstrap,flagsmith-seed)' 2>/dev/null \
  || kubectl get jobs -n mytravels-default | grep flagsmith


In [ ]:
%%bash
echo "=== API logs ==="
kubectl logs -n mytravels-default -l app=api --tail=50

In [ ]:
%%bash
echo "=== Messaging logs ==="
kubectl logs -n mytravels-default -l app=messaging --tail=50

In [ ]:
%%bash
echo "=== MCP logs ==="
kubectl logs -n mytravels-default -l app=mcp --tail=50

In [ ]:
%%bash
echo "=== Web (nginx access/error) logs ==="
kubectl logs -n mytravels-default -l app=web --tail=50

In [ ]:
%%bash
echo "=== Observability stack ==="
for APP in otel-collector prometheus tempo postgres-exporter cadvisor grafana; do
  echo "--- $APP ---"
  kubectl get pods -n mytravels-default -l app=$APP --no-headers 2>/dev/null || echo "(no pods)"
  kubectl logs -n mytravels-default -l app=$APP --tail=15 2>/dev/null | sed 's/^/    /'
  echo ""
done

In [ ]:
%%bash
# Recent events — useful for diagnosing scheduling or PVC binding failures
kubectl get events -n mytravels-default --sort-by='.lastTimestamp' | tail -20

---

## Step 20 — Seed Test Data (Optional)

The stack comes up with an empty database. This step bulk-loads a folder of real photos as points of interest so the Web app, the map, and the Grafana dashboards have something to show.

Each photo is POSTed to `POST /api/pointofinterest/image` as multipart form data, through the Traefik ingress at `http://api.mytravels.local:8080`. The API reads the GPS coordinates from the photo's EXIF metadata, stores the original in MinIO, and publishes to RabbitMQ — which is what drives the messaging worker's thumbnail resize and reverse-geocoding. So this also exercises the full async path end to end, not just the API: the resulting spans show up in Tempo under `service.name=mytravels-messaging`, and the queue depth moves on the RabbitMQ metrics Prometheus scrapes on 15692.

**Photos must be geotagged.** A file whose EXIF carries no GPS data is rejected with HTTP 403 and reported as a failure; the run continues past it to the next file.

Set `PHOTOS_DIR` in the cell below to a folder of photos — non-recursive, matching `.jpg`, `.jpeg`, `.png`, `.heic`, `.heif`. If that folder does not exist the cell skips itself, so this step stays safe when running the notebook top to bottom.

Uploads stream from disk with `curl`, so originals are sent whole — files of 6–16 MB are normal. Kestrel's default request-body limit is 30 MB, so anything larger will come back as HTTP 413.

The loop lives in `../.claude/scripts/upload-photos.sh`; it prints one TAB-separated record per file (name, `OK`/`FAIL`, id or reason) and a `TOTAL` line.

> **Dependencies:** `api.mytravels.local` must resolve (Step 3), the ingress must be applied (Step 17), and RabbitMQ, MinIO, and the messaging worker must be running for the thumbnails and addresses to fill in.
>
> Once the uploads finish the points show up at [http://web.mytravels.local:8080](http://web.mytravels.local:8080) — the `render-config` init container substituted `VITE_API_BASE_URL=http://api.mytravels.local:8080` from the `web-config` ConfigMap into the bundle at pod start (see Step 14), so the browser reaches the API through the same ingress these uploads used.

In [ ]:
%%bash
# Point PHOTOS_DIR at a folder of geotagged photos.
# The cell skips itself if the folder does not exist, so it is safe to run top-to-bottom.
PHOTOS_DIR="${PHOTOS_DIR:-$HOME/Personal/photos}"
API_BASE="http://api.mytravels.local:8080"

if [ ! -d "$PHOTOS_DIR" ]; then
  echo "SKIPPED — not a directory: $PHOTOS_DIR"
  echo "Set PHOTOS_DIR above to a folder of geotagged photos and re-run this cell to seed data."
  exit 0
fi

if ! curl -s -o /dev/null -m 5 "$API_BASE/api/pointofinterest"; then
  echo "API not reachable at $API_BASE — diagnosing inline:"
  echo "--- pods (app=api) ---"
  kubectl get pods -n mytravels-default -l app=api -o wide
  echo "--- ingress ---"
  kubectl get ingress api-ingress -n mytravels-default
  echo "--- api logs (last 30) ---"
  kubectl logs -n mytravels-default -l app=api --tail=30
  exit 1
fi

echo "=== Uploading photos from: $PHOTOS_DIR ==="
../.claude/scripts/upload-photos.sh "$PHOTOS_DIR" "$API_BASE"

In [ ]:
%%bash
echo "=== Points of interest now in the database ==="
curl -s http://api.mytravels.local:8080/api/pointofinterest | python3 -c "
import json, sys
pois = json.load(sys.stdin)
print(f'{len(pois)} point(s) of interest')
for p in pois[:10]:
    desc = 'described' if p.get('description') else 'description pending'
    print(' ', p.get('id'), '|', p.get('formattedAddress') or '(address pending geocoding)', '|', desc, f\"({len(p.get('tags') or [])} tags)\")
if len(pois) > 10:
    print(f'  ... and {len(pois) - 10} more')
" || {
  echo "API not reachable or returned non-JSON — diagnosing inline:"
  kubectl get pods -n mytravels-default -l app=api -o wide
  kubectl logs -n mytravels-default -l app=api --tail=50
}
echo ""
echo "Thumbnails, formatted addresses, and description/tags are all filled in asynchronously by the messaging worker."
echo "Polling the most recently uploaded point for description/tags (calls the Anthropic API — can take up to a minute)..."
LATEST_ID=$(curl -s http://api.mytravels.local:8080/api/pointofinterest | python3 -c "
import json, sys
pois = json.load(sys.stdin)
print(pois[-1]['id'] if pois else '')
")
if [ -n "$LATEST_ID" ]; then
  FOUND=0
  for i in $(seq 1 12); do
    RESULT=$(curl -s http://api.mytravels.local:8080/api/pointofinterest | python3 -c "
import json, sys
pois = json.load(sys.stdin)
poi = next((p for p in pois if p['id'] == $LATEST_ID), None)
if poi and poi.get('description'):
    print(f\"OK id=$LATEST_ID description={poi['description']!r} tags={[t['name'] for t in poi['tags']]}\")
")
    if [ -n "$RESULT" ]; then
      echo "$RESULT"
      FOUND=1
      break
    fi
    sleep 5
  done
  if [ "$FOUND" -eq 0 ]; then
    echo "No description on point $LATEST_ID after 60s — diagnosing inline:"
    echo "--- messaging pods ---"
    kubectl get pods -n mytravels-default -l app=messaging -o wide
    echo "--- messaging logs (last 200, filtered to AppendImageTags) ---"
    kubectl logs -n mytravels-default -l app=messaging --tail=200 | grep -i "AppendImageTags\|append-image-tags" | tail -40
  fi
else
  echo "No points of interest to poll — upload photos first."
fi
echo ""
kubectl logs -n mytravels-default -l app=messaging --tail=15
echo ""
echo "View the points on the map: http://web.mytravels.local:8080"

---

## Step 21 — SOLR Search Verification

POI search runs on Apache SOLR, not PostgreSQL. `GET /api/pointofinterest` (the map's own
list), `GET /api/pointofinterest/search`, and the MCP `search_pointofinterest` tool all resolve
through the `mytravels-pois` collection. The PostgreSQL `ILIKE` search and the
`spGetPointOfInterestByTagName` stored procedure that used to serve them are gone, and so is
`GET /api/pointofinterest/filter` — `/search` covers it.

Two design choices are worth seeing in practice here, because the cells below exercise both:

- **The schema is created at runtime, not mounted.** The container's `solr-precreate` command
  creates an empty collection from SOLR's default configset; the fifteen fields the app queries
  are added by the `SolrSchemaInitializer` hosted service inside `messaging`, which retries with
  backoff until SOLR answers. That is why there is no configset bind mount in Compose and no
  ConfigMap in the Kubernetes stages — one definition, in C#, shared by all five stages.
- **Documents are keyed on `PointOfInterestKey` and written four times per upload.** At creation
  the address, description and tags are all still empty, so `index-solr` is published once up
  front (the POI is immediately findable by date and coordinates) and again as each of those
  fields lands. SOLR upserts by key, so the repeats cost nothing — and it is what makes a full
  rebuild converge on exactly the same document set as incremental indexing.
- **The list endpoint reads the index too.** `GET /api/pointofinterest` is a single capped `*:*`
  SOLR query, so the map and the search box are served from exactly the same documents. Two
  consequences come with that: a freshly uploaded point is absent from the map until the messaging
  worker has consumed its `index-solr` message, and a library larger than `rows` (100 by default)
  is silently truncated — pass `?rows=` to widen it.

| Service | URL | Credentials |
|---|---|---|
| SOLR Admin UI | http://solr.mytravels.local:8080/solr/#/mytravels-pois | — |


In [ ]:
%%bash
SOLR="http://solr.mytravels.local:8080"

echo "=== Collection ping ==="
CODE=$(curl -s -o /tmp/solr_ping.json -w "%{http_code}" --max-time 10 "$SOLR/solr/mytravels-pois/admin/ping?wt=json")
if [ "$CODE" != "200" ]; then
  echo "FAILED — HTTP $CODE. Diagnosing inline:"
  kubectl get pods -n mytravels-default -l app=solr -o wide
  kubectl logs -n mytravels-default -l app=solr --tail=30
  exit 1
fi
python3 -c "import json; print('ping status:', json.load(open('/tmp/solr_ping.json')).get('status'))"

echo ""
echo "=== Fields added by SolrSchemaInitializer (running inside messaging) ==="
curl -s "$SOLR/solr/mytravels-pois/schema/fields" -o /tmp/solr_fields.json
python3 - <<'PY' | tee /tmp/solr_schema_check.txt
import json
want = ["poi_id","formatted_address","tags","tag_exact","tag_ids","description",
        "date_taken","date_created","latitude","longitude","container",
        "original_file_name","generated_blob_name","image_resized","correlation_id"]
have = {f["name"] for f in json.load(open("/tmp/solr_fields.json"))["fields"]}
for name in want:
    print("  OK      " if name in have else "  MISSING ", name)
missing = [n for n in want if n not in have]
print()
print(f"SCHEMA_OK — all {len(want)} fields present" if not missing
      else "SCHEMA_MISSING — " + ", ".join(missing))
PY

if grep -q SCHEMA_MISSING /tmp/solr_schema_check.txt; then
  echo ""
  echo "The schema is applied at startup by SolrSchemaInitializer, which retries with"
  echo "backoff and logs an error rather than crashing if SOLR never appears."
  echo "Its log says what happened:"
  kubectl logs -n mytravels-default -l app=messaging --tail=40 | grep -i -e solr -e schema || \
    kubectl logs -n mytravels-default -l app=messaging --tail=40
fi


In [ ]:
%%bash
SOLR="http://solr.mytravels.local:8080"
API="http://api.mytravels.local:8080"

echo "=== The POI list is served from the index, not from PostgreSQL ==="
INDEXED=$(curl -s "$SOLR/solr/mytravels-pois/select?q=*:*&rows=0" | python3 -c "
import json, sys
print(json.load(sys.stdin)['response']['numFound'])
")
KEYS=$(curl -s "$API/api/pointofinterest" | python3 -c "
import json, sys
print(len({p['pointOfInterestKey'] for p in json.load(sys.stdin)}))
")
echo "  SOLR documents:        $INDEXED"
echo "  keys from the API:     $KEYS   (one document per key, not per row)"

if [ "$INDEXED" != "$KEYS" ]; then
  echo ""
  echo "  The two disagree, which is the rows cap: GET /api/pointofinterest defaults to 100."
  echo "  Re-requesting with rows=$INDEXED:"
  curl -s "$API/api/pointofinterest?rows=$INDEXED" | python3 -c "
import json, sys
print('   ', len({p['pointOfInterestKey'] for p in json.load(sys.stdin)}), 'distinct key(s)')
"
fi

if [ "$KEYS" = "0" ]; then
  echo ""
  echo "SKIPPED the rest — no points of interest yet. Run the seed-test-data step first."
  exit 0
fi

echo ""
echo "=== Search with no term (q=*:*, so everything matches) ==="
curl -s "$API/api/pointofinterest/search?rows=5" | python3 -c "
import json, sys
pois = json.load(sys.stdin)
print(f'{len(pois)} result(s)')
for p in pois:
    tags = ', '.join(t['name'] for t in p.get('tags') or []) or '(no tags yet)'
    print(' ', p.get('id'), '|', p.get('formattedAddress') or '(address pending)', '|', tags)
"

echo ""
echo "=== Free-text search across address, tags and description ==="
echo "    boosts are query-time (formatted_address^5 tags^3 description^1), so relevance"
echo "    can be retuned without reindexing or a schema change"
for TERM in beach city mountain; do
  N=$(curl -s "$API/api/pointofinterest/search?term=$TERM&rows=50" | python3 -c "
import json, sys
print(len(json.load(sys.stdin)))
" 2>/dev/null || echo 0)
  printf "  term=%-10s %s result(s)\n" "$TERM" "$N"
done

echo ""
echo "=== Searching by tag — the path spGetPointOfInterestByTagName used to serve ==="
echo "    /api/pointofinterest/filter is gone; /search covers it, matching the tag through"
echo "    the tokenised tags field that relevance ranks on"
TAG=$(curl -s "$API/api/pointofinterest" | python3 -c "
import json, sys
for p in json.load(sys.stdin):
    for t in p.get('tags') or []:
        print(t['name'])
        raise SystemExit
" 2>/dev/null)
if [ -n "$TAG" ]; then
  echo "  using tag: $TAG"
  curl -s -G "$API/api/pointofinterest/search" --data-urlencode "term=$TAG" -d "rows=50" | python3 -c "
import json, sys
pois = json.load(sys.stdin)
print(' ', len(pois), 'result(s)')
for p in pois[:5]:
    print('   ', p.get('id'), '|', p.get('formattedAddress') or '(address pending)')
"
else
  echo "  SKIPPED — no POI carries tags yet."
  echo "  Tags come from the AppendImageTags subscriber, which needs a working AnthropicApiKey."
fi


In [ ]:
%%bash
SOLR="http://solr.mytravels.local:8080"
API="http://api.mytravels.local:8080"

# Count the documents BEFORE triggering the purge. GET /api/pointofinterest reads SOLR,
# not PostgreSQL, so asking after the worker has purged returns 0 — and the settle loop
# below would match 0 against 0 and report a rebuild that had not happened yet.
EXPECTED=$(curl -s "$API/api/pointofinterest" | python3 -c "
import json, sys
print(len({p['pointOfInterestKey'] for p in json.load(sys.stdin)}))
")
echo "Documents currently indexed: $EXPECTED"
echo ""

echo "=== Triggering a full rebuild (purge, then re-read PostgreSQL) ==="
CORR=$(curl -s -X POST "$API/api/pointofinterest/reindex?purgeFirst=true" | python3 -c "
import json, sys
print(json.load(sys.stdin)['correlationId'])
")
echo "202 Accepted — correlationId: $CORR"
echo ""
echo "The rebuild runs in the messaging worker, never inline in the API, which is why the"
echo "call returns immediately. It is followable on the same correlation id the traceability"
echo "UI groups by:  $API/api/traceability/$CORR"

echo ""
echo "=== Waiting for the rebuild to settle (expecting $EXPECTED document(s)) ==="
for i in $(seq 1 15); do
  N=$(curl -s "$SOLR/solr/mytravels-pois/select?q=*:*&rows=0" | python3 -c "
import json, sys
print(json.load(sys.stdin)['response']['numFound'])
" 2>/dev/null || echo 0)
  echo "  attempt $i: $N indexed"
  [ "$N" = "$EXPECTED" ] && break
  sleep 2
done

echo ""
echo "=== Audit trail for this rebuild ==="
curl -s "$API/api/traceability/$CORR" -o /tmp/solr_reindex_trace.json
python3 -c "
import json
events = json.load(open('/tmp/solr_reindex_trace.json'))
for e in events:
    print(' ', e['createdAt'], e['exchangeName'], e['eventType'], e.get('errorMessage') or '')
if not events:
    print('  (no audit rows yet)')
"

if ! grep -q ConsumeSucceeded /tmp/solr_reindex_trace.json; then
  echo ""
  echo "The rebuild has not reported ConsumeSucceeded. Diagnosing inline:"
  kubectl logs -n mytravels-default -l app=messaging --tail=40 | grep -i -e solr -e schema || \
    kubectl logs -n mytravels-default -l app=messaging --tail=40
fi

echo ""
echo "The rebuild reads spGetPointOfInterest() — latest row per key — and the incremental"
echo "index-solr path resolves the same latest-row-per-key before writing. That equivalence"
echo "is why a purge-and-rebuild is a safe recovery path rather than a divergence risk."


---

## Step 22 — Teardown

Delete all resources and the cluster. Run cells individually to tear down selectively, or run all to wipe everything.

In [ ]:
%%bash
# Delete all manifests in reverse order
kubectl delete -f manifests/9-ingress.yaml
kubectl delete -f manifests/8-traefik-config.yaml
kubectl delete -f manifests/observability/
kubectl delete -f manifests/web/
kubectl delete -f manifests/mcp/
kubectl delete -f manifests/messaging/
kubectl delete -f manifests/api/
kubectl delete -f manifests/flagsmith/
kubectl delete -f manifests/solr/
kubectl delete -f manifests/minio/
kubectl delete -f manifests/rabbitmq/
kubectl delete -f manifests/migrations/
kubectl delete -f manifests/postgres/
kubectl delete -f manifests/1-namespace.yaml

In [ ]:
%%bash
export PATH="/opt/homebrew/bin:/usr/local/bin:$PATH"
# Delete the entire cluster — removes all Docker containers and volumes
k3d cluster delete mytravels